In [3]:
# ==========================================================
#READ DOCUMENTS FROM DATA2 & INSERT INTO PINECONE
# ==========================================================

import os
from dotenv import load_dotenv
from pinecone import Pinecone

from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    BSHTMLLoader,
    Docx2txtLoader
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


# ==========================================================
# LOAD ENVIRONMENT VARIABLES
# ==========================================================

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# Using second index (Task 6)
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME2")


# ==========================================================
# CONNECT TO PINECONE
# ==========================================================

pc = Pinecone(api_key=PINECONE_API_KEY)

index = pc.Index(INDEX_NAME)

print("Connected to Index :", INDEX_NAME)


# ==========================================================
# PROJECT PATH
# ==========================================================

BASE_DIR = r"C:\Users\LENOVO\OneDrive\Desktop\AI_AGRICULTURAL_ADVISOR"

DATA_DIR = os.path.join(BASE_DIR, "Data2")

print("Reading documents from :", DATA_DIR)


# ==========================================================
# DOCUMENT LOADERS
# ==========================================================

LOADERS = {
    ".txt": TextLoader,
    ".pdf": PyPDFLoader,
    ".html": BSHTMLLoader,
    ".docx": Docx2txtLoader
}


# ==========================================================
# CHUNKING
# ==========================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100
)


# ==========================================================
# EMBEDDING MODEL
# ==========================================================

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# ==========================================================
# LOAD DOCUMENTS
# ==========================================================

documents = []

for file in os.listdir(DATA_DIR):

    file_path = os.path.join(DATA_DIR, file)

    if not os.path.isfile(file_path):
        continue

    extension = os.path.splitext(file)[1].lower()

    if extension not in LOADERS:
        print("Skipping :", file)
        continue

    print("Loading :", file)

    if extension == ".html":

        loader = BSHTMLLoader(
            file_path,
            bs_kwargs={"features": "html.parser"}
        )

    else:

        loader = LOADERS[extension](file_path)

    documents.extend(loader.load())


print("\nTotal Documents :", len(documents))


# ==========================================================
# CREATE CHUNKS
# ==========================================================

chunks = splitter.split_documents(documents)

print("Total Chunks :", len(chunks))


# ==========================================================
# CREATE VECTORS
# ==========================================================

vectors = []

for i, chunk in enumerate(chunks):

    embedding = model.encode(
        chunk.page_content
    ).tolist()

    vectors.append({

        "id": f"data2_{i}",

        "values": embedding,

        "metadata": {

            "text": chunk.page_content,

            "source": os.path.basename(
                chunk.metadata.get("source", "")
            ),

            "chunk_id": i

        }

    })

print("Vectors Created :", len(vectors))


# ==========================================================
# INSERT INTO PINECONE
# ==========================================================

BATCH_SIZE = 100

for i in range(0, len(vectors), BATCH_SIZE):

    batch = vectors[i:i+BATCH_SIZE]

    index.upsert(vectors=batch)

    print(f"Uploaded {min(i+BATCH_SIZE, len(vectors))}/{len(vectors)}")


print("\nTASK 6 COMPLETED SUCCESSFULLY")

Connected to Index : agricultural-chatbot
Reading documents from : C:\Users\LENOVO\OneDrive\Desktop\AI_AGRICULTURAL_ADVISOR\Data2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading : ICAR_Kharif_Agro_Advisories_2025.pdf
Loading : ICAR_Maize_Package_of_Practices.pdf
Loading : ICAR_Soil_Nutrient_Management.pdf
Loading : ICAR_Wheat_Cultivation_Pocket_Guide.pdf
Loading : Kisan_Credit_Card_Guidelines.pdf
Loading : NIPHM_IPM_Package_Rice.pdf
Loading : PMFBY_Operational_Guidelines.pdf
Loading : PM_KMY_Operational_Guidelines.pdf
Loading : SoilHealthCard_Implementation_Guidelines.pdf

Total Documents : 722
Total Chunks : 2115
Vectors Created : 2115
Uploaded 100/2115
Uploaded 200/2115
Uploaded 300/2115
Uploaded 400/2115
Uploaded 500/2115
Uploaded 600/2115
Uploaded 700/2115
Uploaded 800/2115
Uploaded 900/2115
Uploaded 1000/2115
Uploaded 1100/2115
Uploaded 1200/2115
Uploaded 1300/2115
Uploaded 1400/2115
Uploaded 1500/2115
Uploaded 1600/2115
Uploaded 1700/2115
Uploaded 1800/2115
Uploaded 1900/2115
Uploaded 2000/2115
Uploaded 2100/2115
Uploaded 2115/2115

TASK 6 COMPLETED SUCCESSFULLY


In [ ]:
# ==========================================================
# SIMILARITY SEARCH
# ==========================================================

from sentence_transformers import SentenceTransformer

# ==========================
# LOAD EMBEDDING MODEL
# ==========================

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# ==========================
# USER INPUT
# ==========================

query = input("Enter your question: ")

print("\nUser Query :", query)

# ==========================
# CREATE QUERY EMBEDDING
# ==========================

query_vector = model.encode(
    query
).tolist()

# ==========================
# SEARCH IN PINECONE
# ==========================

results = index.query(

    vector=query_vector,

    top_k=3,

    include_metadata=True

)

# ==========================
# DISPLAY RESULTS
# ==========================

print("\nTop Similar Chunks\n")

for i, match in enumerate(results["matches"], start=1):

    score = match["score"]

    similarity_percentage = score * 100

    print("=" * 80)

    print(f"Result {i}")

    print("-" * 80)

    print(f"Similarity Score      : {score:.4f}")

    print(f"Similarity Percentage : {similarity_percentage:.2f}%")

    print(f"Source File           : {match['metadata'].get('source', 'Unknown')}")

    print(f"Chunk ID              : {match['metadata'].get('chunk_id', 'N/A')}")

    print("\nRetrieved Chunk\n")

    print(match["metadata"]["text"])

    print("=" * 80)

In [ ]:
# ==========================================================
# RAG (Retrieval Augmented Generation)
# ==========================================================

import os
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


# ==========================================================
# LOAD ENVIRONMENT VARIABLES
# ==========================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


# ==========================================================
# EMBEDDING MODEL
# ==========================================================

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


# ==========================================================
# OPENAI LLM
# ==========================================================

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)


# ==========================================================
# TAKE USER INPUT
# ==========================================================

query = input("Enter your agriculture question: ")

print("\nQuestion :", query)


# ==========================================================
# CREATE QUERY EMBEDDING
# ==========================================================

query_vector = embedding_model.encode(
    query
).tolist()


# ==========================================================
# RETRIEVE FROM PINECONE
# ==========================================================

results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)


# ==========================================================
# DISPLAY RETRIEVED CHUNKS
# ==========================================================

context = ""

print("\nRetrieved Chunks\n")

for i, match in enumerate(results["matches"], start=1):

    metadata = match["metadata"]

    print("=" * 80)
    print(f"Result {i}")
    print("-" * 80)

    print("Similarity Score :", round(match["score"], 4))
    print("Similarity %     :", f"{match['score']*100:.2f}%")
    print("Source File      :", metadata.get("source", "Unknown"))
    print("Chunk ID         :", metadata.get("chunk_id", "N/A"))

    print("\nChunk Content:\n")
    print(metadata["text"])
    print("=" * 80)
    print()

    context += metadata["text"] + "\n\n"


# ==========================================================
# CREATE PROMPT
# ==========================================================

prompt = ChatPromptTemplate.from_template("""

You are an Agricultural AI Assistant.

Answer ONLY from the given context.

If the answer is not found in the context, reply:

"I could not find the answer in the provided documents."

Context:
{context}

Question:
{question}

""")


messages = prompt.format_messages(
    context=context,
    question=query
)


# ==========================================================
# GENERATE ANSWER
# ==========================================================

response = llm.invoke(messages)


# ==========================================================
# DISPLAY FINAL ANSWER
# ==========================================================

print("\n" + "=" * 80)
print("RAG ANSWER")
print("=" * 80)

print(response.content)